Importamos las librerias necesarias para la realización del proceso

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import COLUMNS_TO_DROP, DATA_PATH, TARGET

Se crea el df_score para cargar el data set.
Se crea la variable mermo_depresion en la cual se va a guardar un valor que se va a convertir en entero con .astype.
Si el valor es 1, si hubo cambio es decir que mejoro
Si el valor es 0, No mermo o empeoró.

In [ ]:
df_score = pd.read_csv(DATA_PATH)
df_score = df_score.dropna(subset=['madrs1', 'madrs2']).copy()
df_score[TARGET] = (df_score['madrs2'] < df_score['madrs1']).astype(int)

Seleccionar las variables predictoras (X) eliminando identificadores y las variables del resultado (madrs), madrs1 se mantiene como línea de base inicial, pero madrs2 debe eliminarse para evitar fuga de datos (data leakage).

In [ ]:
X = df_score.drop(columns=[*COLUMNS_TO_DROP, TARGET])
y = df_score[TARGET]

Codificar variables categóricas (como 'age', 'edu', etc.) a numéricas.
Eliminar los valores nulos.

In [ ]:
X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median(numeric_only=True))

Guarda el 30 % de los datos para la evaluación final y el 70% para entrenar. random_state=42: Fija la semilla aleatoria para que siempre obtengas los mismos resultados al ejecutar el código. Asegura que la proporción entre personas que mermaron (1) y no mermaron (0) sea exactamente la misma en el grupo de entrenamiento y en el de prueba.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Se crea un diccionario con los 4 algoritmos listos para ser entrenados bajo la misma semilla aleatoria.

In [ ]:
modelos = {
    'Regresión Logística': LogisticRegression(random_state=42),
    'Máquina de Soporte Vectorial (SVC)': SVC(random_state=42),
    'Árbol de Decisión': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

Recorre los 4 algoritmos uno a uno, entrena el algoritmo con el
70% de los datos.
Evalúa qué porcentaje de casos acertó el modelo (de 0.00 a 1.00)
score_train: Evaluación en los datos con los que estudió.
score_test: Evaluación en los datos no vistos de examen.
resultados.append() / pd.DataFrame(): Guarda las puntuaciones en una lista y las convierte en la tabla comparativa final que se mostró en tu pantalla.


In [ ]:
resultados = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train_scaled, y_train)

    score_train = modelo.score(X_train_scaled, y_train)
    score_test = modelo.score(X_test_scaled, y_test)

    resultados.append({
        'Modelo': nombre,
        'Score Train': round(score_train, 4),
        'Score Test': round(score_test, 4)
    })

df_resultados = pd.DataFrame(resultados)
print(df_resultados)

Gráfica de demostración de las variables más importantes para predecir mejoría.

In [ ]:
# Importancia de características en Random Forest
modelo_rf = modelos['Random Forest']

importances = pd.Series(modelo_rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.tail(7).plot(kind='barh', color='teal')
plt.title('Top Variables más Importantes para Predecir Mejoría')
plt.xlabel('Peso / Importancia Relativa')
plt.ylabel('Variable Predictora')
plt.show()